#Ingestão de Dados

Ingestão de dados brutos na camada Bronze: CSVs de filmes e cotação do dólar via API do Banco Central.

## 1. Configuração

Definição de catálogo, schemas e caminho do volume de landing. Imports e criação do schema Bronze.

In [0]:
catalog = "movies"

bronze_schema_name = "bronze"

silver_schema_name = "silver"

gold_schema_name = "gold"


bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"


landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/Landing"

In [0]:
from pyspark.sql.functions import current_timestamp
import requests
import json
from datetime import datetime, timedelta

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")


DataFrame[]

## 2. Ingestão de CSVs para Tabelas Bronze

Leitura bruta dos arquivos CSV da camada de landing e gravação em modo append com metadado `ingestion_datetime`

In [0]:
files_mapping = {
    "movies_info_TMDB_IMDB.csv": "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv": "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv": "bronze.tb_credits_and_tags",
    "movies_reviews.csv": "bronze.tb_movies_reviews"
}

for file_name, table_name in files_mapping.items():
    file_path = f"{landing_path}/{file_name}"
    
    df_raw = (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("inferSchema", "false")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("multiLine", "true")
        .load(file_path)
    )
    
    df_bronze = df_raw.withColumn("ingestion_datetime", current_timestamp())
    
    # Gravação em formato Delta modo append
    df_bronze.write.format("delta").mode("append").saveAsTable(table_name)
    print(f"Tabela {table_name} ingerida com sucesso.")

Tabela bronze.tb_movies_info ingerida com sucesso.
Tabela bronze.tb_movies_financials ingerida com sucesso.
Tabela bronze.tb_movies_metrics ingerida com sucesso.
Tabela bronze.tb_credits_and_tags ingerida com sucesso.
Tabela bronze.tb_movies_reviews ingerida com sucesso.


## 3. Ingestão da API Cotação Dolar

In [0]:
hoje = datetime.now()
data_fim_default = hoje.strftime("%m-%d-%Y")
data_ini_default = (hoje - timedelta(days=7)).strftime("%m-%d-%Y")

dbutils.widgets.text("data_inicial", data_ini_default, "Data Inicial (MM-DD-AAAA)")
dbutils.widgets.text("data_final", data_fim_default, "Data Final (MM-DD-AAAA)")

data_inicio = dbutils.widgets.get("data_inicio")
data_fim = dbutils.widgets.get("data_fim")

url_bcb = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio}'&@dataFinalCotacao='{data_fim}'"
    f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

response = requests.get(url_bcb)
if response.status_code == 200:
    dados_bcb = response.json().get("value", [])
    if dados_bcb:
        df_cotacao_raw = spark.createDataFrame(dados_bcb)
        df_cotacao_bronze = df_cotacao_raw.withColumn("ingestion_datetime", current_timestamp())
        df_cotacao_bronze.write.format("delta").mode("append").saveAsTable("bronze.tb_cotacao_dolar")
        print("Tabela bronze.tb_cotacao_dolar gravada com sucesso.")
    else:
        print("API retornou sem registros para o período especificado.")
else:
    raise Exception(f"Erro na requisição da API BCB: {response.status_code}")

Tabela bronze.tb_cotacao_dolar gravada com sucesso.
